# 텔레그램 주식정보 채널 언급종목 교차검증 파이프라인
### 텔레그램 수집 → 종목 추출(코스닥 초소형주 필터) → 뉴스·공시 정황 확인 → 거래량·가격 패턴 분석 → 설명 가능한 규칙 판정 + 통계적 교차검증 + (연구 트랙) 상장폐지 XAI 백테스트

원래는 "투자 리딩방 스팸을 이상치로 탐지"하는 프로젝트였고, 이후 "건전한 기업으로 투자금이 흘러가도록 돕는
필터링 시스템"(북극성 B)으로 방향을 넓혔습니다. PRD(v1.3) 갱신에 맞춰 이 노트북도 아래와 같이 다시 정리했습니다.

**용어**: 분석 대상은 `텔레그램 주식정보 채널`로 통일합니다. `리딩방`은 외부 통계·기사 원문을 인용할 때만 씁니다.
본 파이프라인은 특정 채널·기업을 리딩방·불법·사기로 확정하지 않으며, 과거 언급 사건에서 관찰된 패턴과
사용자가 추가로 확인할 기준만 제공합니다.

**설계 원칙**: 이 도구의 목적은 "이상치 탐지 기법을 쓰는 것"이 아니라 **실제로 투자자를 더 잘 보호하는 것**입니다.
그래서 최종 판단은 왜 그렇게 표시됐는지 사람이 검증할 수 있는 **설명 가능한 규칙**이 내리고, 통계적 이상치 탐지(Isolation
Forest)는 규칙이 놓칠 수 있는 패턴을 추가로 훑어보는 **보조 교차검증 신호**로만 사용합니다.

**전체 흐름**
1. 텔레그램 주식정보 채널 수집 (Telethon) — *피처 원천 데이터 수집*
2. 채널 메시지에서 언급 종목 추출 + **코스닥 보통주 시가총액 300억 원 미만으로 분석 대상 축소** — *피처 원천 데이터 수집*
3. 언급 종목 관련 뉴스·공시 정황 확인 (언론사 신뢰도 판정은 하지 않음) — *피처 엔지니어링*
4. 뉴스 버스트·근접중복 탐지 + **복수 채널 신호 집중도**(평소엔 조용하다가 언급 시점에만 신호가 몰리는지) 계산 — *피처 엔지니어링*
5. (부록/P2 실험) 재무·공시 자료로 기업 안정성 참고 정보 계산 — 메인 판정에는 사용하지 않음
6. 거래량 급등 + 코스닥 지수 대비 상대수익률 계산 — *피처 엔지니어링*
7. **설명 가능한 규칙으로 4단계 상태 판정** + (비교 종목이 충분할 때만) Isolation Forest로 보조 교차검증
8. 결과 저장 및 다운로드
9. (연구 트랙) 상장폐지 기업 XAI 백테스트 — 실시간 판정과 분리된 별도 사후 검증

> **상태 분류 (4가지)**: `특이 신호 낮음` / `공시 동반 상승 패턴` / `주의 관찰 요망` / `판단 불가`.
> 어떤 상태도 특정 기업의 불법·사기·부실 여부나 매수·매도 적정성을 판정하지 않습니다.

> **회의 피드백 반영 이력**
> 1. **타겟 범위 축소** — 시장 전체 대신 코스닥 보통주·시가총액 300억 원 미만으로 분석 대상을 좁혔습니다(파일럿 → 점진 확장 전략).
> 2. **복수 채널 신호 집중도** — "여러 정보 소스가 평소엔 조용하다가 특정 상황에만 신호가 몰린다면 그 자체가 위험 신호"라는
>    피드백을 반영해, 거래량·뉴스 신호가 언급 시점에만 좁게 몰려 있는지를 판정 규칙에 넣었습니다.
> 3. **고위험 섹터(바이오·로봇 등) 처리** — 아직 기준을 확정하지 않고, 실제 데이터의 섹터 분포를 먼저 확인한 뒤 결정하기로
>    했습니다(부록 재무 데이터 확인 섹션 참고).
> 4. **상장폐지 기업 XAI 백테스트** — "리딩방 피해로 사라진 기업을 찾아 XAI로 사유를 추적하고, 이진분류 실습을 적용해
>    건전한 시장 형성에 기여하자"는 피드백을 반영해 맨 뒤에 별도 연구 트랙으로 추가했습니다. 이 결과는 활성 종목의
>    실시간 판정에는 반영하지 않습니다.

> **주의사항**
> - 텔레그램 크롤링은 본인 명의 API 자격증명으로 **공개 채널만** 대상으로 진행하세요.
> - 이 노트북은 기본적으로 `DEMO_MODE = True`로 설정되어 있어, 실제 API 자격증명 없이도
>   내장 샘플 데이터로 전체 파이프라인을 바로 실행/테스트할 수 있습니다.
> - 각 상태는 통계적·정황적 의심 신호일 뿐 법적 판단이 아닙니다. 최종 결론은 반드시 사람이 검토해야 합니다.
> - 3~6단계의 뉴스/재무/거래량 데이터는 실제 API 자격증명이 없으면 데모 샘플 데이터로 대체되며, 실제 수치가 아닙니다.


In [ ]:
!pip install -q telethon datasketch openpyxl scikit-learn finance-datareader shap


---
## 0단계. 실행 모드 설정

`DEMO_MODE = True`  → 텔레그램/뉴스/재무 API 호출을 모두 건너뛰고 내장 샘플 데이터로 전체 파이프라인을 테스트
`DEMO_MODE = False` → 실제 Telethon 크롤링 + 실제 뉴스 API 호출 수행 (본인 API 자격증명 필요)


In [ ]:
DEMO_MODE = True  # 실제로 크롤링/뉴스 API를 호출하려면 False로 변경하세요


---
# 1단계. 텔레그램 주식정보 채널 수집

## 1-1. 수집 (Telethon)

my.telegram.org 에서 발급받은 `api_id`/`api_hash`로 공개 채널 메시지를 수집합니다.


In [ ]:
import json, time, asyncio
from getpass import getpass

RAW_PATH = "telegram_raw.jsonl"

if not DEMO_MODE:
    # Colab/Jupyter 커널은 이미 asyncio 이벤트 루프를 돌리고 있어서,
    # telethon.sync의 동기식 `with client:` 문법은 "You must use async with ..." 에러가 납니다.
    # 따라서 진짜 async/await 문법으로 작성합니다.
    from telethon import TelegramClient

    api_id = int(getpass("api_id: "))
    api_hash = getpass("api_hash: ")
    client = TelegramClient("session_investment_spam", api_id, api_hash)

    async def crawl_channel(channel_username, out_path, limit=3000):
        entity = await client.get_entity(channel_username)
        with open(out_path, "a", encoding="utf-8") as f:
            async for msg in client.iter_messages(entity, limit=limit):
                if not msg.text:
                    continue
                record = {
                    "channel": channel_username,
                    "channel_id": entity.id,
                    "message_id": msg.id,
                    "date": msg.date.isoformat(),
                    "sender_id": msg.sender_id,
                    "text": msg.text,
                    "views": getattr(msg, "views", None),
                    "forwards": getattr(msg, "forwards", None),
                    "fwd_from": str(msg.fwd_from) if msg.fwd_from else None,
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        await asyncio.sleep(2)  # rate limit 완화
else:
    print("DEMO_MODE=True → 실제 크롤링은 건너뜁니다. (아래에서 샘플 데이터를 로드합니다)")


In [ ]:
# 실제 크롤링 시 아래 시드 채널 유저네임을 채워서 실행하세요.
SEED_CHANNELS = [
    "gogonero2",
    "aitodaystock",
    "YOUTUBE_INV1",
    "SEDOLSTOCK",
    "CryptoRich_02",
    "CryptoRich33",
    "CryptoRich37",
    "CryptoRich43",
    "coinupkor00",
    "fpt_reviews",
    "mtvpro11",
    "GSoVCrTjCs82ZWE186",
    "SOLOMON_KOR17",
    "daebarkcoincoin"
]

if not DEMO_MODE:
    from telethon.errors import UsernameInvalidError, UsernameNotOccupiedError, FloodWaitError

    failed_channels = []

    async with client:
        for ch in SEED_CHANNELS:
            try:
                await crawl_channel(ch, RAW_PATH)
                print(f"  ✓ {ch} 수집 완료")
            except (UsernameInvalidError, UsernameNotOccupiedError):
                print(f"  ✗ {ch}: 존재하지 않거나 삭제된 채널 (건너뜀)")
                failed_channels.append(ch)
            except FloodWaitError as e:
                print(f"  ✗ {ch}: 요청 제한(FloodWait) {e.seconds}초 대기 필요 — 건너뜀")
                failed_channels.append(ch)
            except Exception as e:
                print(f"  ✗ {ch}: {type(e).__name__} — {e}")
                failed_channels.append(ch)

    print(f"수집 완료 → {RAW_PATH}")
    if failed_channels:
        print(f"실패한 채널 ({len(failed_channels)}개): {failed_channels}")


## 1-2. 채널 확장 (스노우볼 샘플링)

수집된 텍스트에서 `t.me/...` 형태의 채널 링크를 추출해 크롤링 대상을 넓힙니다.


In [ ]:
import re

TME_PATTERN = re.compile(r"t\.me/(joinchat/[\w-]+|\+[\w-]+|[\w_]{5,})")

def extract_new_channels(text):
    return TME_PATTERN.findall(text or "")

if not DEMO_MODE:
    discovered = set()
    with open(RAW_PATH, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            discovered.update(extract_new_channels(rec["text"]))
    new_channels = discovered - set(SEED_CHANNELS)
    print(f"신규 발견 채널 수: {len(new_channels)}")
    # 필요 시 new_channels를 SEED_CHANNELS에 추가해 crawl_channel을 재실행하세요.


## 1-3. 데이터 로드 및 전처리

`DEMO_MODE`에 따라 실제 수집 결과(`telegram_raw.jsonl`) 또는 내장 샘플 데이터를 불러와
정규화 + "추천/시그널 성격" 키워드 매칭을 수행합니다. (도메인 WHOIS·문장 임베딩 등 텔레그램 채널 자체의
스팸 여부 판별용 피처는 새 파이프라인의 목적과 맞지 않아 제거했습니다.)


In [ ]:
import pandas as pd

SAMPLE_DATA = [
    # ── 리딩방 스팸으로 의심되는 패턴 (여러 채널에서 유사 문구 반복) ──
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 1, "date": "2026-07-01T09:00:00",
     "sender_id": 5001, "text": "무료 종목 리딩 받아가세요! 이번주 수익률 인증 92% 선착순 카톡 오픈채팅 open.kakao.com/o/gAbC123",
     "views": 15000, "forwards": 320, "fwd_from": None},
    {"channel": "stock_free_2", "channel_id": 1002, "message_id": 1, "date": "2026-07-01T09:05:00",
     "sender_id": 5002, "text": "무료 종목 리딩방 오픈! 이번주 수익 인증 92% 선착순 마감 임박 open.kakao.com/o/gAbC123",
     "views": 14800, "forwards": 305, "fwd_from": None},
    {"channel": "stock_free_3", "channel_id": 1003, "message_id": 1, "date": "2026-07-01T09:10:00",
     "sender_id": 5003, "text": "급등주 무료 리딩 단톡방 초대 수익인증 92% 선착순 open.kakao.com/o/gAbC123",
     "views": 15200, "forwards": 340, "fwd_from": None},
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 2, "date": "2026-07-02T10:00:00",
     "sender_id": 5001, "text": "타점 잡아드립니다 무료 종목 리딩 단톡방 https://t.me/+xYzAbCd",
     "views": 9000, "forwards": 210, "fwd_from": None},
    {"channel": "stock_free_4", "channel_id": 1004, "message_id": 1, "date": "2026-07-03T11:00:00",
     "sender_id": 5004, "text": "선착순 무료 리딩방 수익 인증 92% open.kakao.com/o/gAbC123 서두르세요",
     "views": 16000, "forwards": 360, "fwd_from": None},
    # ── 실제 종목명을 언급하는 "추천성" 메시지 (2단계 종목 추출 데모용) ──
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 3, "date": "2026-07-05T09:30:00",
     "sender_id": 5001, "text": "현대약품 오늘 상한가 갑니다! 지금 안 사면 후회함, 무료 리딩 신청받아요 open.kakao.com/o/gAbC123",
     "views": 12000, "forwards": 250, "fwd_from": None},
    {"channel": "stock_free_4", "channel_id": 1004, "message_id": 2, "date": "2026-07-05T10:00:00",
     "sender_id": 5004, "text": "SK하이닉스 급등 시작! 지금 진입 타이밍입니다 단톡방 링크 open.kakao.com/o/gAbC123",
     "views": 13000, "forwards": 270, "fwd_from": None},
    {"channel": "CryptoRich33", "channel_id": 1005, "message_id": 1, "date": "2026-07-06T08:00:00",
     "sender_id": 5005, "text": "이더리움 단기 저점 진입 완료 20% 수익 무료 선물 시그널 신청 https://forms.gle/a8s7wVoQ2DeJiVUv7",
     "views": 8000, "forwards": 150, "fwd_from": None},
    # ── 정상적인 일반 대화/뉴스 공유로 추정되는 메시지 ──
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 1, "date": "2026-07-01T08:00:00",
     "sender_id": 6001, "text": "오늘 코스피 지수는 전일 대비 0.4% 상승 마감했습니다. 외국인 순매수 전환.",
     "views": 500, "forwards": 3, "fwd_from": None},
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 2, "date": "2026-07-02T08:00:00",
     "sender_id": 6001, "text": "금일 금통위 기준금리 동결 발표, 시장 예상과 부합.",
     "views": 480, "forwards": 2, "fwd_from": None},
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 3, "date": "2026-07-06T08:00:00",
     "sender_id": 6001, "text": "삼성전자 3분기 실적 발표, 시장 예상치에 부합하는 흐름을 보였습니다.",
     "views": 510, "forwards": 4, "fwd_from": None},
    {"channel": "invest_study_group", "channel_id": 2002, "message_id": 1, "date": "2026-07-02T14:00:00",
     "sender_id": 6002, "text": "재무제표 분석 스터디 이번주 토요일 오후 2시에 진행합니다. 참여 원하시는 분은 댓글 남겨주세요.",
     "views": 120, "forwards": 1, "fwd_from": None},
    {"channel": "invest_study_group", "channel_id": 2002, "message_id": 2, "date": "2026-07-03T14:00:00",
     "sender_id": 6003, "text": "지난주 스터디 자료 공유드립니다. 링크는 곧 올리겠습니다.",
     "views": 110, "forwards": 0, "fwd_from": None},
    {"channel": "personal_diary_ch", "channel_id": 2003, "message_id": 1, "date": "2026-07-04T20:00:00",
     "sender_id": 6004, "text": "오늘 하루도 고생 많으셨습니다. 내일은 더 좋은 하루가 되길 바랍니다.",
     "views": 40, "forwards": 0, "fwd_from": None},
]

if DEMO_MODE:
    df = pd.DataFrame(SAMPLE_DATA)
else:
    rows = []
    with open(RAW_PATH, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)

# 실제 크롤링 데이터에는 미디어만 있고 텍스트가 없는 메시지(NaN)가 섞여 있을 수 있어 공백으로 채움
df["text"] = df["text"].fillna("")

print(f"로드된 메시지 수: {len(df)}")
df.head()


### 크롤링 원본 데이터 다운로드 (선택)

이후 단계에 들어가기 전에, 지금까지 불러온 원본 메시지 데이터를 먼저 CSV/XLSX로
저장하고 다운로드하고 싶다면 아래 셀을 실행하세요.


In [ ]:
RAW_CSV_PATH = "telegram_raw_messages.csv"
RAW_XLSX_PATH = "telegram_raw_messages.xlsx"

df.to_csv(RAW_CSV_PATH, index=False, encoding="utf-8-sig")
df.to_excel(RAW_XLSX_PATH, index=False)
print(f"저장 완료 → {RAW_CSV_PATH}, {RAW_XLSX_PATH} ({len(df)}건)")

try:
    from google.colab import files
    files.download(RAW_CSV_PATH)
    files.download(RAW_XLSX_PATH)
except ImportError:
    print("Colab 환경이 아니므로 자동 다운로드는 건너뜁니다. 파일 탐색기에서 직접 받으세요.")


In [ ]:
# 내부 FDS 가이드라인 3.1 검색어 사전을 그대로 카테고리화 (추천/과장/모집/위험/정상비교)
KEYWORD_CATEGORIES = {
    "추천": [r"추천주", r"오늘의\s*종목", r"매수가", r"목표가", r"손절가", r"익절",
           r"시초가\s*매매", r"종가\s*매매", r"단타", r"포착\s*종목"],
    "과장": [r"상한가", r"급등\s*예상주", r"세력주", r"주포", r"재료\s*공개", r"무조건", r"확실", r"마지막\s*기회"],
    "모집": [r"무료\s*(리딩|종목)", r"VIP\s*방", r"본방\s*입장", r"무료\s*체험", r"수익\s*인증",
           r"승률", r"잔고\s*인증", r"선착순", r"멤버(?:십|쉽)", r"단톡방", r"카톡\s*오픈채팅"],
    "위험": [r"원금\s*보장", r"수익\s*보장", r"손실\s*보전", r"대리매매", r"입금", r"거래소\s*가입"],
    "정상비교": [r"장전\s*브리핑", r"증시\s*일정", r"기업\s*공시", r"산업\s*리포트", r"수급\s*동향", r"실적\s*발표"],
}

def normalize(text):
    text = re.sub(r"[\u200b\uFE0F\u3164]", "", text)  # 제로폭/변형 문자 제거
    return text.strip()

def category_hits(text, category):
    return sum(bool(re.search(p, text or "")) for p in KEYWORD_CATEGORIES[category])

df["text_norm"] = df["text"].map(normalize)
for _category in KEYWORD_CATEGORIES:
    df[f"kw_{_category}"] = df["text_norm"].map(lambda t, c=_category: category_hits(t, c))

df["kw_hits"] = df[[f"kw_{c}" for c in KEYWORD_CATEGORIES]].sum(axis=1)  # 기존 코드 호환용 총합

df[["channel", "text_norm"] + [f"kw_{c}" for c in KEYWORD_CATEGORIES]]

### 참고: 채널 유형 프로파일링 (내부 FDS 가이드라인 2장)

가이드라인은 "리딩형이라는 이유만으로 사기 채널로 분류하지 않는다"고 명시합니다. 그래서 채널을 곧바로
위험/정상으로 나누지 않고, 우선 **A 정상비교군 / B 리딩형 / C 광고·모집형 / D 이상행위 후보군**으로
프로파일링만 해둡니다. 이 라벨은 참고용 설명 자료이며, 최종 `investment_guidance` 판정에는 아직 반영하지
않습니다(판정 로직은 6단계의 설명 가능한 규칙이 그대로 유지).


In [ ]:
channel_profile = df.groupby("channel")[[f"kw_{c}" for c in KEYWORD_CATEGORIES]].sum()

def classify_channel_type(row):
    # 가이드라인 2장 우선순위: 정상비교군 신호가 있고 모집/위험 신호가 없으면 A, 위험 신호가 있으면 D,
    # 모집 신호가 있으면 C, 그 외 추천/과장 신호만 있으면 B로 프로파일링
    if row["kw_정상비교"] > 0 and row["kw_모집"] == 0 and row["kw_위험"] == 0:
        return "A_정상비교군"
    if row["kw_위험"] > 0:
        return "D_이상행위후보군"
    if row["kw_모집"] > 0:
        return "C_광고모집형"
    if row["kw_추천"] > 0 or row["kw_과장"] > 0:
        return "B_리딩형"
    return "미분류"

channel_profile["channel_type"] = channel_profile.apply(classify_channel_type, axis=1)
channel_profile

---
# 2단계. 텔레그램 채널 메시지에서 언급 종목 추출 + 분석 대상 축소

`FinanceDataReader`로 KRX(코스피+코스닥) 상장 종목 전체 리스트를 받아와 사전을 구성합니다. 네트워크로 목록을
못 가져오면(오프라인 환경 등) 데모용 소수 종목으로 자동 대체합니다. 사전 매칭은 여전히 완벽한 개체명 인식(NER)은
아니므로, 짧은 종목명이 긴 종목명의 일부를 잘못 채가지 않도록 간단한 보정도 함께 넣었습니다.

**PRD 변경 사항 (타겟 범위 축소)**: 시장 전체를 다루는 대신 **코스닥 보통주 중 시가총액 300억 원 미만**으로
분석 대상을 좁힙니다. 이는 위험 기준이 아니라, 파일럿 규모로 좁혀서 검증하고 점진적으로 넓히자는 팀 피드백을
반영한 운영 기준이며 확보 데이터에 따라 바뀔 수 있습니다.


In [ ]:
KRX_LISTING_DF = None  # 5단계(부록)에서 시가총액·업종을 재사용할 수 있도록 전역에 보관

def load_krx_stock_dictionary():
    global KRX_LISTING_DF
    import FinanceDataReader as fdr
    krx = fdr.StockListing("KRX")  # 코스피+코스닥+코넥스 전 종목, 컬럼: Code, Name, Market, Marcap, Sector 등
    krx = krx.dropna(subset=["Code", "Name"])
    KRX_LISTING_DF = krx
    return [{"name": n, "ticker": c, "asset_type": "주식"} for n, c in zip(krx["Name"], krx["Code"])]

CRYPTO_DICTIONARY = [
    {"name": "이더리움", "ticker": None, "asset_type": "코인"},
    {"name": "비트코인", "ticker": None, "asset_type": "코인"},
]

DEMO_STOCK_DICTIONARY = [
    {"name": "현대약품", "ticker": "004310", "asset_type": "주식"},
    {"name": "SK하이닉스", "ticker": "000660", "asset_type": "주식"},
    {"name": "삼성전자", "ticker": "005930", "asset_type": "주식"},
]

try:
    STOCK_DICTIONARY = load_krx_stock_dictionary() + CRYPTO_DICTIONARY
    print(f"KRX 상장 종목 {len(STOCK_DICTIONARY) - len(CRYPTO_DICTIONARY)}개 + 코인 {len(CRYPTO_DICTIONARY)}개 사전 로드 완료")
except Exception as e:
    print(f"KRX 종목 리스트 로드 실패({type(e).__name__}: {e}) → 데모용 소수 종목 사전으로 대체합니다.")
    STOCK_DICTIONARY = DEMO_STOCK_DICTIONARY + CRYPTO_DICTIONARY

# 짧은 종목명이 긴 종목명의 일부를 먼저 가로채지 않도록 긴 이름부터 확인
STOCK_DICTIONARY = sorted(STOCK_DICTIONARY, key=lambda item: len(item["name"]), reverse=True)

# 2글자 안팎의 짧은 종목명(예: "테스", "DB", "알트")은 단순 substring 매칭 시 "테스트", "DB증권",
# "알트코인" 같은 흔한 단어·다른 단어의 일부로 잘못 걸리는 경우가 실제 크롤링 데이터에서 다수 확인됐습니다.
# 그래서 매칭된 종목명의 앞뒤가 완성형 한글 음절로 바로 이어지면(=더 큰 단어의 일부일 가능성) 제외하되,
# "테스는", "DB가" 처럼 조사가 자연스럽게 붙는 경우는 정상적인 한국어 문장이므로 허용합니다.
KOREAN_PARTICLES = set("은는이가을를의도만과와에게로부터까지처럼보다이나든지라도조차마저밖에뿐야여")

def _is_word_char(ch):
    return ch.isalnum() or ("가" <= ch <= "힣")

def _has_valid_boundary(text, start, end):
    if start > 0 and _is_word_char(text[start - 1]):
        return False
    if end < len(text):
        next_ch = text[end]
        if _is_word_char(next_ch) and next_ch not in KOREAN_PARTICLES:
            return False
    return True

def extract_recommended_stocks(text):
    text = text or ""
    candidates = []
    for item in STOCK_DICTIONARY:
        name = item["name"]
        start = 0
        while True:
            idx = text.find(name, start)
            if idx == -1:
                break
            if _has_valid_boundary(text, idx, idx + len(name)):
                candidates.append(name)
                break
            start = idx + 1
    # 예: "SK하이닉스"가 매칭되면, 그 안에 포함된 "SK" 같은 짧은 매칭은 제거
    return [name for name in candidates if not any(name != other and name in other for other in candidates)]

df["recommended_stocks"] = df["text_norm"].map(extract_recommended_stocks)

# 메시지 단위 → (채널, 메시지, 종목) 단위로 펼치기
mention_rows = []
for _, row in df.iterrows():
    for stock in row["recommended_stocks"]:
        mention_rows.append({
            "channel": row["channel"],
            "message_id": row["message_id"],
            "date": row["date"],
            "stock_name": stock,
            "kw_hits": row["kw_hits"],
            "text_norm": row["text_norm"],
        })

mentions_df = pd.DataFrame(mention_rows)
print(f"추출된 종목 언급 수: {len(mentions_df)}건 (메시지 {len(df)}건 중)")

if len(mentions_df):
    mention_summary = (
        mentions_df.groupby("stock_name")
        .agg(mention_count=("message_id", "count"), channel_count=("channel", "nunique"),
             first_mentioned=("date", "min"))
        .reset_index()
        .sort_values("mention_count", ascending=False)
    )
else:
    mention_summary = pd.DataFrame(columns=["stock_name", "mention_count", "channel_count", "first_mentioned"])

def _ticker_for_stock(name):
    item = next((it for it in STOCK_DICTIONARY if it["name"] == name), None)
    return item["ticker"] if item else None

if len(mention_summary):
    mention_summary["ticker"] = mention_summary["stock_name"].map(_ticker_for_stock)

# PRD Must-have (타겟 범위 축소): 코스닥 보통주 시가총액 300억 원 미만으로 분석 대상을 좁힙니다. 시장
# 전체가 아니라 파일럿 규모로 좁혀서 검증하고 점진적으로 넓히자는 팀 피드백을 반영한 것으로, 위험 기준이
# 아니라 운영 기준입니다(확보 데이터·검증 결과에 따라 변경 가능).
TARGET_MARKET = "KOSDAQ"
TARGET_MAX_MARKET_CAP = 30_000_000_000  # 300억 원 (현재 시점 시가총액 기준)
PREFERRED_STOCK_NAME_PATTERN = re.compile(r"\d*우[A-Z]?$")

def build_target_universe_tickers():
    if KRX_LISTING_DF is None:
        return None  # KRX 리스트를 못 받아온 경우(오프라인 환경 등) 필터를 적용하지 않음
    universe = KRX_LISTING_DF[
        (KRX_LISTING_DF["Market"] == TARGET_MARKET)
        & (KRX_LISTING_DF["Marcap"] < TARGET_MAX_MARKET_CAP)
        & (~KRX_LISTING_DF["Name"].str.contains(PREFERRED_STOCK_NAME_PATTERN, na=False))
    ]
    return set(universe["Code"])

TARGET_UNIVERSE_TICKERS = build_target_universe_tickers()
if TARGET_UNIVERSE_TICKERS is not None:
    print(f"분석 대상 기준: 코스닥 보통주 시가총액 300억 원 미만 {len(TARGET_UNIVERSE_TICKERS)}개 종목(현재 시점 시가총액 기준)")

if len(mention_summary) and TARGET_UNIVERSE_TICKERS is not None:
    if DEMO_MODE:
        print("DEMO_MODE=True → 데모 샘플은 설명 편의상 대형주를 포함하므로 시가총액 필터를 적용하지 않고 그대로 보여드립니다. "
              "실제 모드(DEMO_MODE=False)에서는 이 시점에 코스닥 보통주 시가총액 300억 원 미만으로 좁혀집니다.")
    else:
        mention_summary["in_target_universe"] = mention_summary["ticker"].isin(TARGET_UNIVERSE_TICKERS)
        out_of_universe = int((~mention_summary["in_target_universe"]).sum())
        if out_of_universe:
            print(f"언급된 종목 중 {out_of_universe}개는 분석 대상(코스닥 보통주 시가총액 300억 원 미만) 밖이라 이후 단계에서 제외합니다.")
        mention_summary = mention_summary[mention_summary["in_target_universe"]].drop(columns=["in_target_universe"]).reset_index(drop=True)

# 실제 크롤링 데이터는 종목 사전이 커서(KRX 전체) 언급 빈도가 낮은 오탐/잡음이 길게 꼬리를 남길 수 있고,
# 이후 단계(뉴스·공시·거래량 조회)가 종목당 여러 번 API를 호출하므로 전체를 다 돌면 지나치게 오래 걸립니다.
# 그래서 실제로 집중적으로 언급된 종목(언급 빈도 상위)만 추려서 이후 단계로 넘깁니다.
TOP_N_STOCKS_FOR_ANALYSIS = 30
if len(mention_summary) > TOP_N_STOCKS_FOR_ANALYSIS:
    print(f"추출된 종목 {len(mention_summary)}개 중 언급 빈도 상위 {TOP_N_STOCKS_FOR_ANALYSIS}개만 이후 단계에서 분석합니다.")
    mention_summary = mention_summary.head(TOP_N_STOCKS_FOR_ANALYSIS).reset_index(drop=True)

mention_summary


---
# 3단계. 추천 종목 관련 언론 기사 조사

네이버 뉴스 검색 API(`https://openapi.naver.com/v1/search/news.json`)로 각 추천 종목의 최근 기사를 조회합니다.
Client ID/Secret은 [네이버 개발자센터](https://developers.naver.com/apps/#/register)에서 애플리케이션 등록 후 발급받습니다.


In [ ]:
import requests
from urllib.parse import quote

SAMPLE_NEWS = [
    # ── "현대약품" 관련 — 동시다발적 보도자료성 기사(작전 의심 패턴) ──
    {"stock_name": "현대약품", "title": "현대약품, 특징주 부각...단기 급등세",
     "description": "현대약품이 특징주로 부각되며 단기 급등세를 보이고 있다. 매수 문의가 폭주하는 모습이다.",
     "pubDate": "2026-07-05T11:00:00", "link": "https://press-release-a.example.com/1"},
    {"stock_name": "현대약품", "title": "현대약품 특징주 부각, 단기 급등세 지속",
     "description": "현대약품이 특징주로 부각되며 단기 급등세를 지속하고 있다. 매수 문의가 폭주하는 상황.",
     "pubDate": "2026-07-05T11:30:00", "link": "https://press-release-b.example.com/1"},
    {"stock_name": "현대약품", "title": "[특징주] 현대약품, 급등세... 매수 문의 폭주",
     "description": "현대약품이 특징주로 부각되며 단기 급등세를 보이는 중이다. 매수 문의가 폭주하고 있다.",
     "pubDate": "2026-07-05T12:00:00", "link": "https://press-release-c.example.com/1"},
    # ── "SK하이닉스" 관련 — 정상적인 실적/업황 기사 (한국경제 = KPF 참여매체) ──
    {"stock_name": "SK하이닉스", "title": "SK하이닉스, HBM 수요 확대에 3분기 실적 개선 전망",
     "description": "증권가는 SK하이닉스의 HBM 수요 확대로 3분기 실적이 시장 예상치를 상회할 것으로 내다봤다.",
     "pubDate": "2026-07-05T09:00:00", "link": "https://www.hankyung.com/article/2026070500001"},
    # ── "삼성전자" 관련 — 정상적인 실적 기사 (매일경제 = KPF 참여매체) ──
    {"stock_name": "삼성전자", "title": "삼성전자 3분기 실적 예상치 부합, 반도체 업황 개선세",
     "description": "삼성전자가 3분기 실적을 발표하며 시장 예상치에 부합하는 흐름을 보였다.",
     "pubDate": "2026-07-06T09:00:00", "link": "https://www.mk.co.kr/article/2026070600001"},
]

def fetch_naver_news(query, display=20, client_id=None, client_secret=None):
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {"X-Naver-Client-Id": client_id, "X-Naver-Client-Secret": client_secret}
    params = {"query": query, "display": display, "sort": "date"}
    resp = requests.get(url, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    items = resp.json().get("items", [])
    return [
        {
            "stock_name": query,
            "title": re.sub("<.*?>", "", it["title"]),
            "description": re.sub("<.*?>", "", it["description"]),
            "pubDate": it["pubDate"],
            "link": it["link"],
        }
        for it in items
    ]

if not DEMO_MODE:
    naver_client_id = getpass("네이버 뉴스 API Client ID: ")
    naver_client_secret = getpass("네이버 뉴스 API Client Secret: ")

    news_rows = []
    for stock in mention_summary["stock_name"]:
        try:
            news_rows.extend(fetch_naver_news(stock, client_id=naver_client_id, client_secret=naver_client_secret))
        except Exception as e:
            print(f"  ✗ {stock} 뉴스 조회 실패: {type(e).__name__} — {e}")
    news_df = pd.DataFrame(news_rows)
else:
    print("DEMO_MODE=True → 실제 뉴스 API 호출을 건너뛰고 샘플 기사 데이터를 사용합니다.")
    news_df = pd.DataFrame(SAMPLE_NEWS)

news_df["press_domain"] = news_df["link"].map(lambda l: re.search(r"https?://([\w.-]+)", l).group(1) if l else None)
print(f"수집된 기사 수: {len(news_df)}건")
news_df


## 3-1. 뉴스 출처 정황 확인
### (PRD 변경: 언론사 신뢰도 "점수" 산정은 Out of Scope)

이전 버전은 한국언론진흥재단(KPF) 참여 언론사 명단을 화이트리스트로 써서 언론사별 "신뢰도 점수"를 매겼습니다.
하지만 참여사 목록이나 자체 큐레이션 목록만으로 매체 신뢰도를 확정할 수 없다는 점을 반영해, **언론사 신뢰도
판정 자체를 이 프로젝트의 범위에서 제외**했습니다(PRD 9번 Out of Scope). 대신 기사 링크에서 도메인만 추출해
참고 정보로 표시하고, "이 기사가 얼마나 몰려서(버스트) 나왔는지·문구가 얼마나 비슷한지"는 4단계의 뉴스
버스트·근접중복 탐지로 확인합니다.


In [ ]:
print("언론사 신뢰도 점수는 계산하지 않습니다 (Out of Scope). 아래는 기사 출처 도메인 참고 정보입니다.")
news_df[["stock_name", "title", "press_domain"]]


## 3-2. 뉴스-주가 상관관계 확인
### (회의 반영: 뉴스-주가 상관관계 — Open DART 실제 연동)

뉴스가 몰렸다고 해서 전부 의심스러운 건 아닙니다. **실제 공시(실적·계약·임상 등)와 맞물려 있다면 정상적인 호재**이고,
공시 없이 뉴스만으로 가격·거래량이 흔들렸다면 세력의 "숫자놀음"에 가깝습니다.

[Open DART](https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS001)에서 무료 인증키를 발급받아
① 전체 상장사의 종목코드↔고유번호 매핑(`corpCode.xml`)을 받고, ② 공시검색 API(`list.json`)로 채널
언급 시점 기준 **전 7일 · 당일 · 후 3일** 구간에 실제 공시가 있었는지 조회합니다(내부 FDS 가이드라인 8.2절).


In [ ]:
import zipfile, io
import xml.etree.ElementTree as ET

DART_CORP_CODE_URL = "https://opendart.fss.or.kr/api/corpCode.xml"
DART_LIST_URL = "https://opendart.fss.or.kr/api/list.json"

# 가이드라인 8.2: 추천 시점 기준 전 7일 · 당일 · 후 3일 구간의 공시를 조회
DISCLOSURE_WINDOW_BEFORE_DAYS = 7
DISCLOSURE_WINDOW_AFTER_DAYS = 3

def load_dart_corp_codes(api_key):
    resp = requests.get(DART_CORP_CODE_URL, params={"crtfc_key": api_key}, timeout=15)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        xml_bytes = zf.read(zf.namelist()[0])
    root = ET.fromstring(xml_bytes)
    rows = [
        {
            "corp_code": corp.findtext("corp_code"),
            "corp_name": corp.findtext("corp_name"),
            "stock_code": (corp.findtext("stock_code") or "").strip(),
        }
        for corp in root.findall("list")
    ]
    return pd.DataFrame([r for r in rows if r["stock_code"]])  # 비상장 법인(종목코드 없음)은 제외

def fetch_dart_disclosures(corp_code, bgn_de, end_de, api_key):
    params = {
        "crtfc_key": api_key, "corp_code": corp_code,
        "bgn_de": bgn_de, "end_de": end_de,
        "page_no": 1, "page_count": 100,
    }
    resp = requests.get(DART_LIST_URL, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    return data.get("list", []) if data.get("status") == "000" else []

if not DEMO_MODE:
    dart_api_key = getpass("Open DART 인증키: ")
    corp_code_df = load_dart_corp_codes(dart_api_key)
    ticker_to_corp_code = corp_code_df.set_index("stock_code")["corp_code"].to_dict()

    # 우선주(예: "대덕1우", "두산2우B")는 DART 관점에서 별도 법인이 아니라 보통주를 발행한 법인과 동일한
    # 법인입니다. DART corpCode.xml은 그 법인의 대표 종목코드(보통주)로만 등록돼 있어, 우선주 티커로는
    # 절대 매핑되지 않습니다. 이름에서 우선주 표기를 떼어 보통주 이름으로 되돌린 뒤 재조회합니다.
    PREFERRED_STOCK_SUFFIX_PATTERN = re.compile(r"\d*우[A-Z]?$")

    def find_common_stock_ticker(stock_name):
        base_name = PREFERRED_STOCK_SUFFIX_PATTERN.sub("", stock_name)
        if base_name == stock_name:
            return None  # 애초에 우선주 표기가 아니었음
        match = next((item for item in STOCK_DICTIONARY if item["name"] == base_name), None)
        return match["ticker"] if match else None

    def resolve_corp_code(stock_name, ticker):
        if ticker and ticker in ticker_to_corp_code:
            return ticker_to_corp_code[ticker]
        common_ticker = find_common_stock_ticker(stock_name)
        if common_ticker and common_ticker in ticker_to_corp_code:
            return ticker_to_corp_code[common_ticker]
        return None

    disclosure_rows = []
    for _, row in mention_summary.iterrows():
        stock_name = row["stock_name"]
        stock_info = next((item for item in STOCK_DICTIONARY if item["name"] == stock_name), None)
        ticker = stock_info["ticker"] if stock_info else None
        corp_code = resolve_corp_code(stock_name, ticker)

        has_disclosure = False
        if corp_code:
            mention_date = pd.to_datetime(row["first_mentioned"])
            bgn_de = (mention_date - pd.Timedelta(days=DISCLOSURE_WINDOW_BEFORE_DAYS)).strftime("%Y%m%d")
            end_de = (mention_date + pd.Timedelta(days=DISCLOSURE_WINDOW_AFTER_DAYS)).strftime("%Y%m%d")
            try:
                disclosures = fetch_dart_disclosures(corp_code, bgn_de, end_de, dart_api_key)
                has_disclosure = len(disclosures) > 0
            except Exception as e:
                print(f"  ✗ {stock_name} 공시 조회 실패: {type(e).__name__} — {e}")
        else:
            print(f"  · {stock_name}: DART 법인 매핑 없음(코인이거나 비상장 등) → 공시 없음으로 처리")

        disclosure_rows.append({"stock_name": stock_name, "has_official_disclosure": has_disclosure})

    disclosure_check = pd.DataFrame(disclosure_rows)
else:
    print("DEMO_MODE=True → 실제 DART 조회를 건너뛰고 샘플 공시 데이터를 사용합니다.")
    SAMPLE_DISCLOSURES = {
        "현대약품": False,   # 뉴스 버스트 기간에 매칭되는 공식 공시 없음 → 뉴스만으로 만든 움직임 의심
        "SK하이닉스": True,  # 실적 관련 공시 존재
        "삼성전자": True,
    }
    disclosure_check = pd.DataFrame({
        "stock_name": list(SAMPLE_DISCLOSURES.keys()),
        "has_official_disclosure": list(SAMPLE_DISCLOSURES.values()),
    })

disclosure_check

---
# 4단계. 뉴스 버스트·근접중복 탐지 + 복수 채널 신호 집중도

정상적인 기업 뉴스라면 매체마다 문구가 다르고 실적·공시 같은 사실관계를 전달합니다. 반면 채널發 펌핑을
뒷받침하려는 보도자료성 기사는 **여러 매체가 짧은 기간에 거의 동일한 문구를 반복 게시**하는 경향이 있습니다.
이 "근접중복 반복성"은 원래 리딩방 스팸 탐지에 쓰던 MinHash 기법을 그대로 재사용해 잡아냅니다.

**PRD 변경 사항 (팀 피드백 반영)**: "여러 정보 소스가 평소엔 조용하다가 특정 상황(급등)에만 신호가 몰려
있다면 그 자체를 위험 신호로 봐야 한다"는 피드백을 반영해, 뉴스가 평소엔 전혀 없다가 언급 시점 부근에만
좁게 몰려 있는지를 `news_signal_concentrated`로 별도 계산합니다. 6단계에서 거래량 신호와 결합해
`multi_channel_signal_concentration`(복수 채널 신호 집중)을 만듭니다. 여기서 계산한 `pump_news_score`·
`news_price_mismatch`·`news_signal_concentrated`는 6단계에서 설명 가능한 규칙의 입력으로 쓰입니다.
언론사 신뢰도는 더 이상 입력으로 쓰지 않습니다(3-1단계 참고).


In [ ]:
from datasketch import MinHash, MinHashLSH

PUMP_PR_KEYWORDS = [r"특징주", r"급등세", r"매수\s*문의\s*폭주", r"단기\s*급등", r"테마\s*부각"]

def get_minhash(text, num_perm=64, shingle_size=4):
    # 한국어는 조사·어미가 붙어 공백 기준 단어 분리가 불안정하므로,
    # 공백을 제거한 문자 단위 n-gram(shingle)으로 유사도를 비교합니다.
    text = "".join(text.split())
    m = MinHash(num_perm=num_perm)
    shingles = {text[i:i + shingle_size] for i in range(max(len(text) - shingle_size + 1, 1))}
    for sh in shingles:
        m.update(sh.encode("utf8"))
    return m

def pr_keyword_hits(text):
    return sum(bool(re.search(p, text or "")) for p in PUMP_PR_KEYWORDS)

news_df["combined_text"] = (news_df["title"].fillna("") + " " + news_df["description"].fillna(""))
news_df["pr_keyword_hits"] = news_df["combined_text"].map(pr_keyword_hits)

lsh = MinHashLSH(threshold=0.3, num_perm=64)
duplicate_count = []
for idx, text in enumerate(news_df["combined_text"]):
    mh = get_minhash(text)
    matches = lsh.query(mh)
    duplicate_count.append(len(matches))
    lsh.insert(str(idx), mh)
news_df["duplicate_count"] = duplicate_count

# 텔레그램 채널에서 해당 종목이 처음 언급된 시점 대비, N일 이내 몰린 기사 수(버스트) 계산
BURST_WINDOW_DAYS = 3
first_mention = mention_summary.set_index("stock_name")["first_mentioned"] if len(mention_summary) else pd.Series(dtype="object")

def days_since_mention(row):
    if row["stock_name"] not in first_mention.index:
        return None
    mention_date = pd.to_datetime(first_mention[row["stock_name"]])
    news_date = pd.to_datetime(row["pubDate"])
    return (news_date - mention_date).total_seconds() / 86400

news_df["days_since_mention"] = news_df.apply(days_since_mention, axis=1)
news_df["is_burst"] = news_df["days_since_mention"].between(0, BURST_WINDOW_DAYS)

def normalize01(series):
    s = series.astype(float)
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng else s * 0

stock_news = news_df.groupby("stock_name").agg(
    article_count=("title", "count"),
    avg_duplicate_count=("duplicate_count", "mean"),
    total_pr_keyword_hits=("pr_keyword_hits", "sum"),
    burst_count=("is_burst", "sum"),
).reset_index()

# 회의 반영: 공시 없이 뉴스만으로 몰린 경우("뉴스-주가 상관관계" 미스매치)를 별도 신호로 포함
stock_news = stock_news.merge(disclosure_check, on="stock_name", how="left")
stock_news["has_official_disclosure"] = stock_news["has_official_disclosure"].fillna(False)
stock_news["news_price_mismatch"] = (stock_news["burst_count"] > 0) & (~stock_news["has_official_disclosure"])

# 팀 피드백 반영: 뉴스가 "평소엔 없다가 언급 시점에만 좁게 몰리는" 패턴(복수 채널 신호 집중의 뉴스 축).
# article_count가 전부 burst 구간에서 나왔다면(=버스트 밖 기간엔 기사가 전혀 없었다면) 신호가 좁게 집중된 것으로 봅니다.
stock_news["news_signal_concentrated"] = (
    (stock_news["burst_count"] > 0) & (stock_news["burst_count"] == stock_news["article_count"])
)

# 보조 배지: 뉴스 정황을 사람이 바로 읽을 수 있는 문구로 정리 (PRD 9번 Scope "보조 배지" 참고)
def news_badges(row):
    if row["article_count"] == 0:
        return ["뉴스 또는 공시 데이터 부족"]
    badges = ["관찰 기간 내 공시 확인" if row["has_official_disclosure"] else "관찰 기간 내 관련 가능 공시 미확인"]
    if row["avg_duplicate_count"] >= 1:
        badges.append("유사 기사 반복 관찰")
    if row["total_pr_keyword_hits"] > 0:
        badges.append("보도자료성 표현 관찰")
    if row["news_signal_concentrated"]:
        badges.append("복수 채널 신호 집중(언급 시점에만 몰림)")
    if len(badges) == 1:  # 공시 배지 외에 다른 특이 신호가 없으면
        badges.append("뉴스 정황 특이 신호 낮음")
    return badges

stock_news["news_badges"] = stock_news.apply(news_badges, axis=1)

# pump_news_score: 언론사 신뢰도는 더 이상 입력값으로 쓰지 않습니다(Out of Scope). 근접중복·보도자료성
# 표현·버스트·뉴스-공시 불일치·복수 채널 신호 집중만으로 재계산합니다.
stock_news["pump_news_score"] = (
    0.3 * normalize01(stock_news["avg_duplicate_count"])
    + 0.2 * normalize01(stock_news["total_pr_keyword_hits"])
    + 0.25 * normalize01(stock_news["burst_count"])
    + 0.15 * stock_news["news_price_mismatch"].astype(float)
    + 0.1 * stock_news["news_signal_concentrated"].astype(float)
)

stock_news.sort_values("pump_news_score", ascending=False)[[
    "stock_name", "article_count", "avg_duplicate_count", "total_pr_keyword_hits",
    "burst_count", "has_official_disclosure", "news_price_mismatch",
    "news_signal_concentrated", "pump_news_score", "news_badges",
]]


---
# 부록. 선택적 재무 데이터 확인 (P2 연구 실험 — 메인 판정에는 사용하지 않음)

Open DART로 부채비율(재무제표 주요계정)·감사의견·관리종목 지정 이력(공시 목록에서 검색)·최대주주 지분율
변동을 조회하고, 시가총액·업종은 2단계에서 이미 받아온 KRX 상장 종목 리스트(`KRX_LISTING_DF`)를 재사용합니다.

**PRD 변경 사항**: 이 섹션에서 계산하는 `stability_score`/`vulnerability_score`는 더 이상 6단계 최종 판정
(`investment_guidance`)에 반영하지 않습니다. 업종·기업 특성에 따라 해석이 달라 전체 종목에 동일 기준으로
우량·부실을 판정할 수 없고, 오판·낙인 가능성이 크다는 점을 반영한 결정입니다(PRD 9번 Out of Scope 참고).
이 값들은 참고용 부가 정보로만 남겨둡니다.

바이오·로봇 등 고위험 섹터를 어떻게 별도로 다룰지는 아직 확정하지 않았습니다 — 이 섹션에서 실제로
수집되는 데이터의 섹터 분포를 먼저 확인한 뒤 결정합니다(PRD 11번 Open Issues 참고).


In [ ]:
DART_FNLTT_URL = "https://opendart.fss.or.kr/api/fnlttSinglAcnt.json"
DART_AUDIT_OPINION_URL = "https://opendart.fss.or.kr/api/accnutAdtorNmNdAdtOpinion.json"
DART_MAJOR_HOLDER_CHANGE_URL = "https://opendart.fss.or.kr/api/hyslrChgSttus.json"

DEFAULT_REPRT_CODE = "11011"  # 사업보고서(연간) — 반기/분기가 필요하면 11012/11013/11014로 교체
DEFAULT_BSNS_YEAR = str(pd.Timestamp.now().year - 1)  # 가장 최근 확정된 사업연도

def fetch_dart_json(url, params):
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    return data.get("list", []) if data.get("status") == "000" else []

def fetch_debt_ratio(corp_code, api_key, bsns_year=DEFAULT_BSNS_YEAR, reprt_code=DEFAULT_REPRT_CODE):
    rows = fetch_dart_json(DART_FNLTT_URL, {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code": reprt_code,
    })
    accounts = {r.get("account_nm"): r.get("thstrm_amount") for r in rows}
    debt, equity = accounts.get("부채총계"), accounts.get("자본총계")
    if debt is None or equity is None:
        return None
    debt, equity = float(str(debt).replace(",", "")), float(str(equity).replace(",", ""))
    return (debt / equity * 100) if equity else None

def fetch_audit_opinion(corp_code, api_key, bsns_year=DEFAULT_BSNS_YEAR, reprt_code=DEFAULT_REPRT_CODE):
    rows = fetch_dart_json(DART_AUDIT_OPINION_URL, {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code": reprt_code,
    })
    return rows[0].get("adt_opinion") if rows else None  # 예: "적정"

def fetch_major_holder_change_pct(corp_code, api_key):
    rows = fetch_dart_json(DART_MAJOR_HOLDER_CHANGE_URL, {"crtfc_key": api_key, "corp_code": corp_code})
    if not rows:
        return 0.0
    latest = rows[0]
    try:
        before = float(str(latest.get("bfr_hold_pct", "0") or "0").replace(",", ""))
        after = float(str(latest.get("trmend_hold_pct", "0") or "0").replace(",", ""))
        return after - before
    except (TypeError, ValueError):
        return 0.0

def fetch_administrative_issue(corp_code, api_key, lookback_days=730):
    end_de = pd.Timestamp.now().strftime("%Y%m%d")
    bgn_de = (pd.Timestamp.now() - pd.Timedelta(days=lookback_days)).strftime("%Y%m%d")
    disclosures = fetch_dart_disclosures(corp_code, bgn_de, end_de, api_key)  # 3-2단계에서 정의됨
    return int(any("관리종목" in (d.get("report_nm") or "") for d in disclosures))

def lookup_krx_field(ticker, column):
    if KRX_LISTING_DF is None or ticker is None:
        return None
    row = KRX_LISTING_DF[KRX_LISTING_DF["Code"] == ticker]
    if row.empty or column not in row.columns:
        return None
    return row.iloc[0][column]

def guess_sector_label(ticker):
    # 실제 업종분류가 필요하면 KRX 업종코드 매핑으로 교체해야 합니다. 여기서는 KRX 리스팅의 업종 텍스트에서
    # "바이오/제약/의약품" 키워드가 있는지로 근사치만 추정합니다.
    for column in ("Sector", "Industry"):
        value = lookup_krx_field(ticker, column)
        if value and re.search(r"바이오|제약|의약품", str(value)):
            return "바이오"
    return "일반"

if not DEMO_MODE:
    fundamentals_rows = []
    for stock in mention_summary["stock_name"]:
        stock_info = next((item for item in STOCK_DICTIONARY if item["name"] == stock), None)
        ticker = stock_info["ticker"] if stock_info else None
        corp_code = resolve_corp_code(stock, ticker)  # 3-2단계에서 정의됨 (우선주 → 보통주 법인으로 매핑)

        if not corp_code:
            print(f"  · {stock}: DART 법인 매핑 없음(코인/비상장 등) → 중립값으로 처리")
            fundamentals_rows.append({
                "stock_name": stock, "sector": "미확인", "market_cap_billion": None,
                "debt_ratio": 100.0, "is_administrative_issue": 0,
                "audit_opinion": "확인불가", "major_holder_change_pct": 0.0,
            })
            continue

        try:
            debt_ratio = fetch_debt_ratio(corp_code, dart_api_key)
        except Exception as e:
            print(f"  ✗ {stock} 부채비율 조회 실패: {type(e).__name__} — {e}")
            debt_ratio = None
        try:
            audit_opinion = fetch_audit_opinion(corp_code, dart_api_key)
        except Exception as e:
            print(f"  ✗ {stock} 감사의견 조회 실패: {type(e).__name__} — {e}")
            audit_opinion = None
        try:
            is_admin_issue = fetch_administrative_issue(corp_code, dart_api_key)
        except Exception as e:
            print(f"  ✗ {stock} 관리종목 이력 조회 실패: {type(e).__name__} — {e}")
            is_admin_issue = 0
        try:
            holder_change = fetch_major_holder_change_pct(corp_code, dart_api_key)
        except Exception as e:
            print(f"  ✗ {stock} 최대주주 지분 변동 조회 실패: {type(e).__name__} — {e}")
            holder_change = 0.0

        market_cap = lookup_krx_field(ticker, "Marcap")
        fundamentals_rows.append({
            "stock_name": stock,
            "sector": guess_sector_label(ticker),
            "market_cap_billion": (market_cap / 1e8) if market_cap else None,  # 억원 단위로 환산
            "debt_ratio": debt_ratio if debt_ratio is not None else 100.0,  # 조회 실패 시 중립값
            "is_administrative_issue": is_admin_issue,
            "audit_opinion": audit_opinion if audit_opinion is not None else "확인불가",
            "major_holder_change_pct": holder_change,
        })
        print(f"  ✓ {stock} 조회 완료 ({len(fundamentals_rows)}/{len(mention_summary)})")
        time.sleep(0.3)  # DART 호출량 제한 완화

    fundamentals_df = pd.DataFrame(fundamentals_rows)
else:
    print("DEMO_MODE=True → 실제 DART/KRX 조회를 건너뛰고 샘플 재무 데이터를 사용합니다.")
    SAMPLE_FUNDAMENTALS = [
        {"stock_name": "현대약품", "sector": "바이오", "market_cap_billion": 180, "debt_ratio": 145.0,
         "is_administrative_issue": 0, "audit_opinion": "한정", "major_holder_change_pct": -8.5},
        {"stock_name": "SK하이닉스", "sector": "반도체", "market_cap_billion": 135000, "debt_ratio": 38.0,
         "is_administrative_issue": 0, "audit_opinion": "적정", "major_holder_change_pct": 0.1},
        {"stock_name": "삼성전자", "sector": "반도체", "market_cap_billion": 400000, "debt_ratio": 27.0,
         "is_administrative_issue": 0, "audit_opinion": "적정", "major_holder_change_pct": 0.0},
    ]
    fundamentals_df = pd.DataFrame(SAMPLE_FUNDAMENTALS)

# market_cap_billion 결측치(코인 등)는 극단적으로 취급되지 않도록 관측된 값의 중앙값으로 채움
if fundamentals_df["market_cap_billion"].notna().any():
    fundamentals_df["market_cap_billion"] = fundamentals_df["market_cap_billion"].fillna(
        fundamentals_df["market_cap_billion"].median()
    )
else:
    fundamentals_df["market_cap_billion"] = fundamentals_df["market_cap_billion"].fillna(1.0)

# 부채비율이 높을수록, 관리종목일수록, 감사의견이 '적정'이 아닐수록, 최대주주 지분이 급변할수록 불안정 → 감점
fundamentals_df["debt_risk"] = normalize01(fundamentals_df["debt_ratio"])
fundamentals_df["audit_risk"] = (fundamentals_df["audit_opinion"] != "적정").astype(float)
fundamentals_df["holder_change_risk"] = normalize01(fundamentals_df["major_holder_change_pct"].abs())

fundamentals_df["stability_score"] = 1 - (
    0.4 * fundamentals_df["debt_risk"]
    + 0.3 * fundamentals_df["is_administrative_issue"]
    + 0.2 * fundamentals_df["audit_risk"]
    + 0.1 * fundamentals_df["holder_change_risk"]
)

# 회의 반영: 소액으로도 주가가 흔들리는 "취약 기업" 특성 — 시가총액이 작을수록, 바이오 섹터일수록 소액 개입에 취약
fundamentals_df["small_cap_risk"] = normalize01(1 / fundamentals_df["market_cap_billion"])
fundamentals_df["sector_risk"] = (fundamentals_df["sector"] == "바이오").astype(float)
fundamentals_df["vulnerability_score"] = (
    0.6 * fundamentals_df["small_cap_risk"] + 0.4 * fundamentals_df["sector_risk"]
)

print("참고: stability_score/vulnerability_score는 연구용 참고 정보이며, 6단계 최종 판정(investment_guidance)에는 사용하지 않습니다.")
fundamentals_df[[
    "stock_name", "sector", "market_cap_billion", "debt_ratio", "is_administrative_issue",
    "audit_opinion", "stability_score", "vulnerability_score",
]]

---
# 6단계. 거래량 급등 + 코스닥 지수 대비 상대수익률

"거래량이 평소 대비 300% 이상 급등"하는 현상 자체가 고전적인 이상치 탐지 문제입니다. `FinanceDataReader`로
KRX 실제 일별 시세를 받아와 20일 이동평균 대비 당일 거래량 배율을 계산합니다.

**PRD 변경 사항**: 시장 전체가 같이 오른 건지 개별 종목만 특이하게 움직인 건지 구분하기 위해, 같은 기간
**코스닥 지수(KQ11) 대비 상대수익률**도 함께 계산합니다. 7단계 종합 판단에서는 부록의 재무 취약도 대신
이 상대수익률과 거래량 배율을 설명 가능한 규칙의 입력으로 사용합니다.


In [ ]:
import numpy as np

VOLUME_LOOKBACK_DAYS = 40  # 20거래일 이동평균 계산에 필요한 여유(주말·공휴일 감안)
KOSDAQ_INDEX_TICKER = "KQ11"

def fetch_real_volume_history(ticker, mention_date):
    import FinanceDataReader as fdr
    start = (mention_date - pd.Timedelta(days=VOLUME_LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    end = pd.Timestamp.now().strftime("%Y-%m-%d")
    ohlcv = fdr.DataReader(ticker, start, end)
    if ohlcv is None or ohlcv.empty or "Close" not in ohlcv.columns or "Volume" not in ohlcv.columns:
        return None
    ohlcv = ohlcv.copy()
    ohlcv.index.name = "Date"
    ohlcv = ohlcv.reset_index()
    ohlcv["price_change_pct"] = ohlcv["Close"].pct_change() * 100
    ohlcv["day"] = range(len(ohlcv))
    return ohlcv[["day", "Date", "Volume", "price_change_pct"]].rename(columns={"Volume": "volume"})

def fetch_index_return(mention_date):
    # 코스닥 지수 대비 상대수익률 계산용 — 시장 전체가 오른 건지 개별 종목만 특이하게 움직인 건지 구분
    import FinanceDataReader as fdr
    start = (mention_date - pd.Timedelta(days=VOLUME_LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    end = pd.Timestamp.now().strftime("%Y-%m-%d")
    idx = fdr.DataReader(KOSDAQ_INDEX_TICKER, start, end)
    if idx is None or idx.empty or "Close" not in idx.columns:
        return None
    idx = idx.copy()
    idx.index.name = "Date"
    idx = idx.reset_index()
    idx["index_change_pct"] = idx["Close"].pct_change() * 100
    return idx[["Date", "index_change_pct"]]

if not DEMO_MODE:
    volume_frames = []
    index_hist_cache = {}
    for _, row in mention_summary.iterrows():
        stock = row["stock_name"]
        ticker = row.get("ticker") or next((it["ticker"] for it in STOCK_DICTIONARY if it["name"] == stock), None)

        if not ticker:
            print(f"  · {stock}: 시세 데이터 없음(코인 등) → 거래량 급등 판단에서 제외")
            continue
        try:
            mention_date = pd.to_datetime(row["first_mentioned"])
            hist = fetch_real_volume_history(ticker, mention_date)
            if hist is None or hist.empty:
                print(f"  ✗ {stock}: 시세 데이터를 가져오지 못했습니다.")
                continue
            hist = hist.copy()
            hist["stock_name"] = stock

            mention_key = mention_date.strftime("%Y-%m-%d")
            if mention_key not in index_hist_cache:
                index_hist_cache[mention_key] = fetch_index_return(mention_date)
            index_hist = index_hist_cache[mention_key]
            if index_hist is not None:
                hist = hist.merge(index_hist, on="Date", how="left")
            else:
                hist["index_change_pct"] = None

            volume_frames.append(hist)
        except Exception as e:
            print(f"  ✗ {stock} 시세 조회 실패: {type(e).__name__} — {e}")

    volume_df = (
        pd.concat(volume_frames, ignore_index=True) if volume_frames
        else pd.DataFrame(columns=["day", "Date", "volume", "price_change_pct", "stock_name", "index_change_pct"])
    )
else:
    print("DEMO_MODE=True → 실제 시세 조회를 건너뛰고 샘플 거래량/가격/지수 데이터를 사용합니다.")
    # 데모용 샘플 일별 거래량/가격/지수 데이터 (실제 수치 아님)
    np.random.seed(42)
    volume_rows = []
    for stock in fundamentals_df["stock_name"]:
        base_volume = np.random.randint(500_000, 2_000_000)
        for day in range(20):
            volume = int(base_volume * np.random.uniform(0.8, 1.2))
            price_change_pct = np.random.uniform(-2, 2)
            index_change_pct = np.random.uniform(-0.5, 0.5)  # 코스닥 지수는 개별 종목보다 변동폭이 작다고 가정
            volume_rows.append({
                "stock_name": stock, "day": day, "volume": volume,
                "price_change_pct": price_change_pct, "index_change_pct": index_change_pct,
            })
        # 마지막 날, "현대약품"만 인위적으로 거래량 급등 + 시장 대비 초과 상승(채널 언급 직후 상황을 재현)
        if stock == "현대약품":
            volume_rows[-1]["volume"] = int(base_volume * 4.2)
            volume_rows[-1]["price_change_pct"] = 18.5
            volume_rows[-1]["index_change_pct"] = 0.1
    volume_df = pd.DataFrame(volume_rows)

def compute_volume_ratio(group):
    group = group.sort_values("day").copy()
    rolling_avg = group["volume"].rolling(window=19, min_periods=5).mean().shift(1)
    group["volume_ratio"] = group["volume"] / rolling_avg
    return group

if len(volume_df):
    volume_df = pd.concat(
        [compute_volume_ratio(g) for _, g in volume_df.groupby("stock_name")],
        ignore_index=True,
    )
    volume_df["volume_spike_flag"] = volume_df["volume_ratio"] >= 3.0  # 300% 이상 급등 (참고용 표시, 최종 판정은 다음 셀의 규칙이 담당)
    # 코스닥 지수 대비 상대수익률: 시장 전체가 같이 오른 건지, 개별 종목만 튄 건지 구분
    volume_df["relative_return"] = volume_df["price_change_pct"] - volume_df["index_change_pct"]
    latest_volume = volume_df.sort_values("day").groupby("stock_name").tail(1)
else:
    latest_volume = pd.DataFrame(columns=[
        "stock_name", "day", "volume", "volume_ratio", "price_change_pct",
        "index_change_pct", "relative_return", "volume_spike_flag",
    ])

latest_volume[[
    "stock_name", "day", "volume", "volume_ratio", "price_change_pct",
    "index_change_pct", "relative_return", "volume_spike_flag",
]]


## 종합 판단: 설명 가능한 규칙이 최종 결정, 통계적 이상치는 교차검증만

2~4단계·6단계에서 만든 피처(`pump_news_score`, `news_price_mismatch`, `news_signal_concentrated`,
`volume_ratio`, `relative_return`)로 **최종 판단은 사람이 검증 가능한 규칙**이 내립니다. 부록의 재무
데이터(`stability_score`/`vulnerability_score`)는 여기서 사용하지 않습니다. 비교할 종목이 충분히 쌓였을
때만 Isolation Forest로 "규칙이 놓쳤을 수 있는 통계적 이상치"를 추가로 훑어보되, 이 신호가 규칙의 최종
판단을 뒤집지는 않습니다.

**상태 분류 (4가지, PRD 9번 Scope 참고)**

| 상태 | 의미 |
| --- | --- |
| `특이 신호 낮음` | 거래량 급증과 시장 대비 큰 하락이 함께 관찰되지 않음. 안전/정상이라는 뜻은 아님 |
| `공시 동반 상승 패턴` | 거래량 변화는 있었으나 시장 대비 큰 하락은 없고, 관찰 기간 내 관련 가능 공시가 확인됨 |
| `주의 관찰 요망` | 거래량 급증 + 시장 대비 큰 하락이 함께 관찰됨, 또는 평소엔 조용하다가 언급 시점에만 거래량·뉴스 신호가 좁게 몰림(복수 채널 신호 집중). 불법·사기 판정이 아니라 추가 확인이 필요한 과거 패턴을 뜻함 |
| `판단 불가` | 가격·거래량·시장지수 등 핵심 데이터가 부족함 |


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

final_df = (
    mention_summary
    .merge(
        stock_news[["stock_name", "pump_news_score", "news_price_mismatch", "news_signal_concentrated",
                     "has_official_disclosure", "news_badges"]],
        on="stock_name", how="left",
    )
    .merge(fundamentals_df[["stock_name", "sector"]], on="stock_name", how="left")
    .merge(
        latest_volume[["stock_name", "volume_ratio", "price_change_pct", "index_change_pct",
                        "relative_return", "volume_spike_flag"]],
        on="stock_name", how="left",
    )
)

final_df["pump_news_score"] = final_df["pump_news_score"].fillna(0)
final_df["news_price_mismatch"] = final_df["news_price_mismatch"].fillna(False)
final_df["news_signal_concentrated"] = final_df["news_signal_concentrated"].fillna(False)
final_df["has_official_disclosure"] = final_df["has_official_disclosure"].fillna(False)
final_df["volume_spike_flag"] = final_df["volume_spike_flag"].fillna(False)

# 팀 피드백 반영: 거래량 신호도 "평소엔 없다가 언급 시점에만 몰렸는지"를 함께 봅니다. 여기서는 언급 시점의
# 거래량 급증(volume_spike_flag)과 뉴스 신호 집중(news_signal_concentrated)이 함께 나타나는지로 근사합니다.
final_df["multi_channel_signal_concentration"] = final_df["volume_spike_flag"] & final_df["news_signal_concentrated"]

# 시장 대비 큰 하락 기준 — 데이터가 더 쌓이면 재조정이 필요한 실험값입니다(PRD 10번 Constraints 참고).
RELATIVE_RETURN_DROP_THRESHOLD = -10.0  # 코스닥 지수 대비 상대수익률(%p)이 이 값 이하면 "시장 대비 큰 하락"

# ── 최종 판단: 설명 가능한 규칙 (투자자 보호 도구는 "왜 표시했는지" 사람이 검증할 수 있어야 함) ──
# 재무 건전성(부채비율 등)은 부록의 P2 실험일 뿐이라 여기서는 사용하지 않습니다. 시가총액 300억 원 미만
# 필터로 이미 "소액 개입에 취약한" 종목군으로 분석 대상을 좁혔으므로, 임계값도 고정값을 씁니다.
def classify(row):
    has_core_data = pd.notna(row["volume_ratio"]) and pd.notna(row["relative_return"])
    if not has_core_data:
        return "판단 불가"

    volume_spike = bool(row["volume_spike_flag"])
    big_underperform = row["relative_return"] <= RELATIVE_RETURN_DROP_THRESHOLD
    signal_concentrated = bool(row["multi_channel_signal_concentration"])

    if volume_spike and (big_underperform or signal_concentrated):
        return "주의 관찰 요망"
    if volume_spike and not big_underperform and row["has_official_disclosure"]:
        return "공시 동반 상승 패턴"
    return "특이 신호 낮음"

final_df["investment_guidance"] = final_df.apply(classify, axis=1)

# 사람이 바로 읽을 수 있는 근거 배지 — 뉴스 배지(4단계) + 복수 채널 신호 집중 여부
def combine_badges(row):
    badges = list(row["news_badges"]) if isinstance(row["news_badges"], list) else []
    if row["multi_channel_signal_concentration"] and "복수 채널 신호 집중(언급 시점에만 몰림)" not in badges:
        badges.append("복수 채널 신호 집중(언급 시점에만 몰림)")
    return badges

final_df["badges"] = final_df.apply(combine_badges, axis=1)

# ── 보조 교차검증: Isolation Forest — 규칙이 놓쳤을 수 있는 통계적 이상 패턴을 추가로만 훑어봅니다.
#     최종 결정을 뒤집지 않으며, 비교 종목이 너무 적으면(5개 미만) 통계적으로 무의미해 아예 건너뜁니다.
FEATURE_COLS = [
    "pump_news_score", "news_price_mismatch", "multi_channel_signal_concentration",
    "volume_ratio", "price_change_pct", "relative_return",
]

if len(final_df) >= 5:
    feature_df = final_df[FEATURE_COLS].copy()
    for col in ("news_price_mismatch", "multi_channel_signal_concentration"):
        feature_df[col] = feature_df[col].astype(float)
    feature_df = feature_df.fillna(feature_df.median(numeric_only=True))

    X_scaled = StandardScaler().fit_transform(feature_df.values)
    contamination = min(0.3, max(0.05, 3 / len(final_df)))
    iso = IsolationForest(n_estimators=300, contamination=contamination, random_state=42)
    iso.fit(X_scaled)
    final_df["stat_outlier_score"] = -iso.score_samples(X_scaled)
    final_df["stat_outlier_flag"] = iso.predict(X_scaled) == -1
else:
    final_df["stat_outlier_score"] = None
    final_df["stat_outlier_flag"] = False
    print(f"비교 대상 종목이 {len(final_df)}개뿐이라 통계적 교차검증(Isolation Forest)은 신뢰도가 낮아 건너뜁니다. "
          f"(종목이 5개 이상 쌓이면 자동으로 활성화됩니다)")

final_df[[
    "stock_name", "sector", "mention_count", "pump_news_score", "news_price_mismatch",
    "volume_ratio", "relative_return", "multi_channel_signal_concentration",
    "investment_guidance", "badges", "stat_outlier_score", "stat_outlier_flag",
]]


### 참고 지표: 사전 신호 노출 비율 (백테스트용, 확정 성과 아님)

> PRD 6번 Success Metrics의 "Optional Backtest" 항목에 해당하는 참고 계산입니다. 실제 성과로 확정하지 않고,
> 거래량이 급등한 종목 중 몇 개가 사전에 `주의 관찰 요망`으로 표시되어 있었는지만 봅니다.
>
> `사전 신호 노출 비율 = (급등 + 주의 관찰 요망 종목 수) / (전체 거래량 급등 종목 수) × 100`
>
> 정답 라벨(실제 불법·사기 여부)이 없는 상태의 참고용 계산이며, 9단계(연구 트랙) 상장폐지 XAI 백테스트가
> 이 한계를 보완하기 위한 별도 시도입니다.


In [ ]:
spiked = final_df[final_df["volume_spike_flag"]]
if len(spiked):
    prior_signal_rate = (spiked["investment_guidance"] == "주의 관찰 요망").mean() * 100
    print(f"거래량 급등 종목 {len(spiked)}개 중 '주의 관찰 요망' 표시 비율(참고 지표): {prior_signal_rate:.1f}%")
else:
    print("이번 데이터에서는 거래량 급등(300%+)이 감지된 종목이 없습니다.")


---
# 7단계. 결과 저장 및 다운로드


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(final_df["stock_name"], final_df["pump_news_score"], label="pump_news_score")
plt.bar(final_df["stock_name"], final_df["volume_ratio"].fillna(0) / 10, alpha=0.6, label="volume_ratio (÷10)")
plt.title("종목별 뉴스 의심도 vs 거래량 배율")
plt.ylabel("score")
plt.legend()
plt.tight_layout()
plt.show()

OUT_CSV_PATH = "investment_guidance_result.csv"
OUT_XLSX_PATH = "investment_guidance_result.xlsx"

# CSV/XLSX는 리스트 값을 직접 저장할 수 없으므로, 배지 컬럼(리스트)은 저장 직전에만 문자열로 변환합니다.
LIST_COLUMNS_TO_STRINGIFY = ["badges", "news_badges"]
final_df_export = final_df.copy()
for col in LIST_COLUMNS_TO_STRINGIFY:
    if col in final_df_export.columns:
        final_df_export[col] = final_df_export[col].map(lambda b: "; ".join(b) if isinstance(b, list) else b)

final_df_export.to_csv(OUT_CSV_PATH, index=False, encoding="utf-8-sig")
final_df_export.to_excel(OUT_XLSX_PATH, index=False)
print(f"결과 저장 완료 → {OUT_CSV_PATH}, {OUT_XLSX_PATH}")

try:
    from google.colab import files
    files.download(OUT_CSV_PATH)
    files.download(OUT_XLSX_PATH)
except ImportError:
    print("Colab 환경이 아니므로 자동 다운로드는 건너뜁니다. 파일 탐색기에서 직접 받으세요.")


---
## 한계 및 유의사항

### 1~2단계: 수집·종목 추출·분석 대상 축소
- `DEMO_MODE=True`일 때의 샘플 데이터(메시지·뉴스·재무·거래량)는 전부 파이프라인 동작 확인용 예시이며 실제 수치가 아닙니다.
- 종목 추출은 KRX 전체 상장 종목 사전(FinanceDataReader)으로 확장했는데, 실제 대량 크롤링 데이터에서
  짧은 종목명이 다른 단어의 일부로 잘못 매칭되는 문제가 확인돼 경계 검사를 추가했지만, 완벽한 한국어
  개체명 인식(NER)은 아니라 여전히 일부 오탐/누락이 남을 수 있습니다.
- **코스닥 보통주 시가총액 300억 원 미만** 필터를 추가했습니다. 이는 위험 기준이 아니라 시장 전체 대신
  파일럿 규모로 좁혀서 검증하자는 팀 피드백을 반영한 운영 기준이며, 데이터 분포에 따라 조정이 필요합니다.
  필터는 실행 시점의 시가총액을 기준으로 적용되므로, 실제 언급 시점의 시가총액과 다를 수 있습니다(사건
  당시 시총 보완은 TODO).
- `DEMO_MODE`에서는 설명 편의상 대형주(SK하이닉스·삼성전자 등) 샘플을 그대로 두고 시가총액 필터를 건너뜁니다
  — 실제 모드에서만 필터가 실질적으로 적용됩니다.
- KRX 종목 리스트는 실행할 때마다 새로 받아오므로, 실행 시점에 따라 상장폐지·신규상장이 반영되어 사전 내용이
  달라질 수 있습니다.

### 3단계: 뉴스·공시
- **언론사 신뢰도 "점수" 산정은 이 프로젝트의 범위에서 제외했습니다.** 참여사 목록이나 자체 큐레이션 목록만으로
  매체 신뢰도를 확정할 수 없다는 판단에 따른 것입니다(PRD 9번 Out of Scope). 기사 링크의 도메인은 참고
  정보로만 표시합니다.
- 네이버 뉴스 검색 API는 호출량 제한이 있고 언론사 정보를 명시적으로 주지 않아 URL 도메인으로 추정합니다.
- 공시 매칭은 실제 Open DART API를 씁니다. 코인·비상장 종목은 DART 매핑이 아예 없어 항상 "공시 없음"으로
  처리됩니다 — 이건 데이터 누락이지 실제 의심 신호가 아니므로 해석 시 주의가 필요합니다.
- **우선주**는 DART에 별도 법인이 없고 보통주 발행 법인과 동일한 법인이라, 종목명에서 우선주 표기를 떼어
  보통주 이름으로 재조회하도록 처리했지만, 이름 패턴 가정이 100% 보장되진 않습니다.

### 4단계: 뉴스 버스트·복수 채널 신호 집중
- "작전세력 의심 보도" 판별은 근접중복·버스트 타이밍 등 통계적 의심 신호일 뿐입니다. 실제 허위·과장 보도인지는
  기사 원문 검증과 사람의 판단이 반드시 필요합니다.
- **복수 채널 신호 집중**(`multi_channel_signal_concentration`)은 "거래량 급증 + 뉴스가 언급 시점에만 몰려
  있고 그 외 기간엔 기사가 전혀 없음"으로 근사했습니다. 팀 피드백("평소엔 조용하다가 특정 상황에만 신호가
  몰리면 위험 신호")을 반영한 초기 설계이며, 관찰 기간·"평소" 기준의 타당성은 실제 데이터로 검증이 필요합니다
  (PRD 11번 Open Issues).

### 부록(P2): 재무 데이터
- `stability_score`/`vulnerability_score`는 참고용으로만 계산하며, 6단계 최종 판정에는 사용하지 않습니다.
- 업종(바이오 여부)은 KRX 리스팅 텍스트에서 키워드로 근사 추정한 것이라 부정확할 수 있습니다.
- 바이오·로봇 등 고위험 섹터의 실제 표본 내 비중을 아직 확인하지 못했습니다. 비중이 높게 나타나면 별도
  패턴 분석·전용 배지를 추가하는 방향을 검토합니다(PRD 11번 Open Issues).

### 6단계: 거래량·코스닥 지수 대비 상대수익률·최종 판정
- 거래량 급등은 FinanceDataReader로 실제 KRX 일별 시세를 가져와 계산합니다(20일 이동평균 대비 배율). 상장폐지·
  거래정지·최근 상장 종목은 이력이 부족해 이동평균 계산이 불안정할 수 있습니다.
- 코스닥 지수 대비 상대수익률은 종목별 언급 시점마다 별도로 지수를 조회해 병합하며, 지수 데이터를 못 가져오면
  `판단 불가`로 처리됩니다.
- 최종 판정은 사람이 검증 가능한 규칙이 내리고, Isolation Forest는 비교 종목이 5개 이상일 때만 보조
  교차검증 신호로 참고합니다.
- 규칙 임계값(거래량 3.0배, 상대수익률 -10%p 등)은 초기 설계일 뿐 검증된 값이 아니므로, 실제 사례가 쌓이면
  재조정이 필요합니다.
- 어떤 상태도 특정 기업의 불법·사기·부실 여부나 매수·매도 적정성을 판정하지 않습니다.

### 참고 지표 (구 "북극성" 지표)
- 급등 종목 중 사전 `주의 관찰 요망` 노출 비율은 참고 계산일 뿐이며, 정답 라벨(실제 불법·사기 여부) 없이
  계산한 것이라 확정 성과로 사용하지 않습니다.

### 9단계 (연구 트랙): 상장폐지 XAI 백테스트
- 지금은 **DEMO 샘플(가상 데이터)** 로만 파이프라인 구조를 시연합니다. 실제 상장폐지-텔레그램 언급 매칭
  데이터셋은 아직 구축되지 않았습니다(TODO, PRD 11번 Open Issues).
- XAI(SHAP 또는 순열 중요도) 결과는 상관관계 설명이며 인과관계 증명이 아닙니다. "리딩방 피해 때문에
  상장폐지됐다"고 확정하지 않습니다.
- 상장폐지된 종목만 표본에 넣으면 생존편향이 발생하므로, 음성 표본(언급됐지만 상장폐지되지 않은 종목)을
  반드시 함께 포함해야 합니다.
- 이 섹션의 결과는 활성 종목의 실시간 판정에 반영하지 않습니다.


---
# 9단계 (연구 트랙). 상장폐지 기업 XAI 백테스트

**팀 피드백 반영**: "리딩방 피해로 사라진(상장폐지된) 기업을 찾아 XAI로 분석하면 어떤 사유로 사라졌는지
추적할 수 있고, 교육과정에서 배운 이진분류·XAI 실습을 프로젝트에 적용하면서 기업이 망하지 않고 유지되는
건전한 시장에 기여할 수 있다"는 의견을 반영했습니다.

이 섹션은 **1~8단계의 실시간 판정 파이프라인과 분리된 별도 연구용 백테스트**입니다. 목적은 실제
상장폐지(1)/유지(0) 라벨이 있는 과거 사례로 이진분류 모델을 학습시키고, XAI(설명가능 AI)로 어떤 신호가
상장폐지와 가장 강하게 연관됐는지 살펴보는 것입니다. 결과는 연구용 참고 정보이며, 활성 종목의 실시간
상태값(`investment_guidance`)에는 반영하지 않습니다.

**실제 데이터 확보 방법 (TODO, PRD 11번 Open Issues 참고)**
1. [KRX KIND 상장폐지종목 조회](https://kind.krx.co.kr/investwarn/delcompany.do?method=searchDelCompanyMain)에서
   상장폐지 종목·사유·일자를 가져온다.
2. 위 목록을 이 노트북이 실제로 수집한 텔레그램 언급 이력(`mention_summary` 등)과 종목코드 기준으로 매칭한다.
3. 언급은 있었지만 상장폐지되지 않은 종목(생존편향 방지용 음성 표본)도 함께 포함한다.

아직 (1)~(3)을 자동으로 수행하는 코드는 없습니다 — 조사 결과 "이 상장폐지가 리딩방 피해 때문이었다"를
종목 단위로 1:1 확정해주는 공식 목록은 찾지 못했고(PRD 2번 Background 참고), 상장폐지 사유는 감사의견
거절·매출액 미달·자본잠식·시세조종 등으로 다양해 KIND 사유만으로는 인과관계를 확정할 수 없습니다.
지금은 `XAI_BACKTEST_DEMO_MODE = True`로 파이프라인 구조만 시연합니다.


In [ ]:
# ============================================================
# 9단계 (연구 트랙): 상장폐지 기업 XAI 백테스트 — 데이터 구성
# ============================================================
XAI_BACKTEST_DEMO_MODE = True  # 실제 매칭 데이터가 준비되면 False로 바꾸고 아래 데이터 로드 부분을 교체하세요.

# 샘플 라벨 데이터(가상의 값, 실제 상장폐지 사례가 아닙니다) — 기존 파이프라인이 산출하는 피처와 같은
# 종류(거래량 배율, 시장 대비 상대수익률, 뉴스 근접중복, 뉴스 버스트, 공시 유무, 복수 채널 신호 집중)를
# 사용해, delisted=1(상장폐지) 표본은 "언급 시점에 거래량·뉴스가 몰리고 이후 시장 대비 큰 하락, 공시
# 없음"에 가깝게, delisted=0(유지) 표본은 반대 방향에 가깝게 생성한 가상의 데모용 분포입니다.
np.random.seed(7)

def _make_backtest_sample(n, delisted):
    if delisted:
        volume_ratio = np.random.uniform(2.5, 6.0, n)
        relative_return = np.random.uniform(-40, -10, n)
        avg_duplicate_count = np.random.uniform(1.0, 3.0, n)
        burst_count = np.random.randint(2, 6, n)
        has_official_disclosure = np.random.choice([0, 1], n, p=[0.8, 0.2])
        news_signal_concentrated = np.random.choice([0, 1], n, p=[0.3, 0.7])
    else:
        volume_ratio = np.random.uniform(0.5, 2.0, n)
        relative_return = np.random.uniform(-8, 15, n)
        avg_duplicate_count = np.random.uniform(0.0, 0.8, n)
        burst_count = np.random.randint(0, 2, n)
        has_official_disclosure = np.random.choice([0, 1], n, p=[0.3, 0.7])
        news_signal_concentrated = np.random.choice([0, 1], n, p=[0.85, 0.15])
    return pd.DataFrame({
        "volume_ratio": volume_ratio,
        "relative_return": relative_return,
        "avg_duplicate_count": avg_duplicate_count,
        "burst_count": burst_count,
        "has_official_disclosure": has_official_disclosure,
        "news_signal_concentrated": news_signal_concentrated,
        "delisted": delisted,
    })

if XAI_BACKTEST_DEMO_MODE:
    backtest_df = pd.concat([
        _make_backtest_sample(25, delisted=1),
        _make_backtest_sample(25, delisted=0),
    ], ignore_index=True).sample(frac=1, random_state=7).reset_index(drop=True)
    print(f"DEMO 샘플 라벨 데이터 {len(backtest_df)}건 (상장폐지 {int(backtest_df['delisted'].sum())}건 / "
          f"유지 {int((backtest_df['delisted'] == 0).sum())}건) — 실제 상장폐지 사례가 아닌 가상 데이터입니다.")
else:
    raise NotImplementedError(
        "실제 상장폐지-텔레그램 언급 매칭 데이터셋 구축은 아직 구현되지 않았습니다. "
        "KRX KIND 상장폐지종목 조회 결과와 mention_summary를 직접 매칭하는 코드를 추가한 뒤 사용하세요."
    )

backtest_df.head()


In [ ]:
# ============================================================
# 9단계 (연구 트랙): 이진분류 모델 + XAI(설명가능 AI)
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance

BACKTEST_FEATURE_COLS = [
    "volume_ratio", "relative_return", "avg_duplicate_count",
    "burst_count", "has_official_disclosure", "news_signal_concentrated",
]

X = backtest_df[BACKTEST_FEATURE_COLS]
y = backtest_df["delisted"]

# 표본이 매우 작아 학습/검증 분리가 통계적으로 불안정할 수 있습니다 — 실데이터 확보 후 표본 수를 늘려야 합니다.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7, stratify=y)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test), target_names=["유지(0)", "상장폐지(1)"]))

# XAI: 가능하면 SHAP으로 피처 기여도를 계산하고, SHAP을 쓸 수 없는 환경이면 순열 중요도(permutation
# importance)로 대체합니다. 둘 다 "이 피처가 예측에 얼마나 기여했는가"를 보여주는 설명 가능성 도구이며,
# 상관관계를 보여줄 뿐 인과관계를 증명하지 않습니다.
try:
    import shap
    explainer = shap.LinearExplainer(clf, X_train)
    shap_values = explainer(X_test)
    xai_importance = pd.Series(
        np.abs(shap_values.values).mean(axis=0), index=BACKTEST_FEATURE_COLS
    ).sort_values(ascending=False)
    xai_method = "SHAP (mean |shap value|)"
except ImportError:
    print("shap 패키지를 사용할 수 없어 순열 중요도(permutation importance)로 대체합니다.")
    perm = permutation_importance(clf, X_test, y_test, n_repeats=30, random_state=7)
    xai_importance = pd.Series(perm.importances_mean, index=BACKTEST_FEATURE_COLS).sort_values(ascending=False)
    xai_method = "Permutation Importance"

print(f"\nXAI 피처 기여도 ({xai_method}) — 상장폐지(1)/유지(0) 이진분류 모델 기준")
xai_importance


### XAI 백테스트 해석 시 유의사항

- 위 결과는 **DEMO 샘플(가상 데이터)** 기준입니다. 실제 상장폐지 사례와 텔레그램 언급 이력을 매칭하기 전까지는
  참고용 파이프라인 구조 시연 이상의 의미가 없습니다.
- 상장폐지 사유는 감사의견 거절·매출액 미달·자본잠식·시세조종 등으로 다양하며, 이 모델의 피처 기여도가
  높다고 해서 "리딩방 피해가 상장폐지의 원인"이라고 확정할 수 없습니다. 상관관계이지 인과관계가 아닙니다.
- 상장폐지된 종목만 표본에 넣으면 생존편향이 발생하므로, 언급됐지만 상장폐지되지 않은 종목(음성 표본)을
  반드시 함께 포함해야 합니다.
- 이 섹션의 결과는 활성 종목의 `investment_guidance` 실시간 판정에 반영하지 않습니다. 순수 연구/백테스트
  목적입니다.
